# MonIA FramePack Kaggle benchmark
Candidate-only I2V benchmark. No narrative authority, no auto-publish to live game.
Probe revision 2 — trigger isolated Kaggle capability run.\n

In [ ]:
!git clone -q --depth 1 https://github.com/lllyasviel/FramePack.git /kaggle/working/FramePack
%cd /kaggle/working/FramePack
%pip -q install -r requirements.txt
print('FramePack installed')


In [ ]:
# This benchmark intentionally uses the official FramePack implementation rather than reimplementing its sampler.
import torch
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2) if torch.cuda.is_available() else 0)
assert torch.cuda.is_available()
print('MONIA_FRAMEPACK_READY')


In [ ]:
import os, shutil
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
os.environ['HF_HOME']='/kaggle/working/hf'
shutil.rmtree('/root/.cache/huggingface', ignore_errors=True)
from pathlib import Path
demo=Path('/kaggle/working/FramePack/demo_gradio_f1.py')
s=demo.read_text()
needle='block.launch('
assert needle in s
inject=r'''# MONIA_HEADLESS_FRAMEPACK
import urllib.request, shutil, json
from PIL import Image as _MoniaImage
_src='/kaggle/working/marion-framepack-source.png'
urllib.request.urlretrieve('https://cdn.openart.ai/openart-uploads/production/attachment-transfers/20f4695a631aec334010eacc4c21c17cc0be959070b4d2c5eeafdce55c564573.png', _src)
_img=np.array(_MoniaImage.open(_src).convert('RGB'))
_prompt='Photorealistic young woman in her warm apartment in natural morning sunlight, seated on the sofa holding a white mug. She hears a quiet sound near the balcony, naturally turns her head and upper body toward it, lowers the mug slightly, shifts her posture as if preparing to stand. Subtle breathing, natural blinking, realistic human body mechanics, restrained everyday movement. External invisible cinematic camera, stable medium-wide framing. She never touches, holds, addresses, or looks into the camera. Preserve her face, hair, cream knit sweater, grey trousers, apartment, plants, balcony and lighting exactly from the reference image. No dialogue.'
_last=None
for _event in process(_img,_prompt,'',221101,3.0,9,25,1.0,10.0,0.0,6.0,False,16):
    if isinstance(_event,tuple) and _event and isinstance(_event[0],str) and _event[0].endswith('.mp4'): _last=_event[0]
assert _last and os.path.isfile(_last), 'FramePack produced no MP4'
_out=Path('/kaggle/working/monia-framepack-output'); _out.mkdir(exist_ok=True)
shutil.copy2(_last,_out/'shot-01.mp4'); shutil.copy2(_src,_out/'source.png')
(_out/'result.json').write_text(json.dumps({'jobId':'marion-framepack-f1-001','state':'candidate','candidateOnly':True,'narrativeAuthority':False,'router':'framepack-f1','clips':['shot-01.mp4'],'humanApprovalRequired':True},indent=2))
print('MONIA_FRAMEPACK_VIDEO_READY',_last)
raise SystemExit(0)
'''
demo.write_text(s.replace(needle,inject+'\n'+needle,1))
!cd /kaggle/working/FramePack && python demo_gradio_f1.py
